# 01 - Cloud YOLO Dataset Preparation



This notebook owns the data side of the team workflow:



1. Configure and authenticate Google Cloud.

2. Sync source images from GCS.

3. Retrieve YOLO labels from Label Studio, GCS, or a JupyterLab upload.

4. Validate image/label parity and YOLO rows.

5. Create deterministic train, validation, and test splits.

6. Write `dataset.yaml`, `split_manifest.json`, and `prep_summary.json`.

7. Inspect labeled samples before training.



Run this notebook whenever source images, labels, classes, exclusions, or split policy changes. Then open `02_cloud_yolo_training.ipynb`, point it at the same `workspace/<project>` directory, and train without downloading or reshuffling data again.



> Authentication is interactive or environment-based; credentials are never stored in the notebook. Raw downloads remain unchanged under `workspace/raw`.



## 1. Notebook setup and dependencies



Use the `Python (yolo-cloud)` kernel created by `install_cloud_workstation.sh`. Uncomment the next line only when packages are missing from the selected kernel.

In [ ]:
# %pip install -U pyyaml requests pillow matplotlib google-cloud-storage



import hashlib

import json

import math

import os

import random

import shutil

import subprocess

import sys

import zipfile

from collections import Counter

from datetime import datetime, timezone

from pathlib import Path



import matplotlib.pyplot as plt

import requests

import yaml

from PIL import Image, ImageDraw



print(f"Python: {sys.version.split()[0]}")

## 2. Project, bucket, and path configuration



Keep project-specific values in this cell. `LABEL_SOURCE_MODE` controls the label workflow:



- `label_studio_api`: request a fresh YOLO export from Label Studio, then optionally mirror it to GCS.

- `gcs_export`: download an existing Label Studio YOLO export archive or directory from GCS.

- `user_upload_zip`: use a YOLO export ZIP uploaded through the JupyterLab file browser. Set `USER_LABEL_ZIP` to that local file.



For Label Studio, set secrets outside the notebook, for example in a terminal: `export LABEL_STUDIO_API_TOKEN="..."`. Do not commit tokens, service-account keys, `.env` files, datasets, or model weights.

In [ ]:
PROJECT_NAME = "serdp_yolo_v0"



GCP_PROJECT_ID = "REPLACE_WITH_GCP_PROJECT_ID"

GCP_REGION = "us-central1"



IMAGE_BUCKET_URI = "gs://nmfs-dev-uc1-pifsc/SFM03/SERDP/Trial/dataset_v0/Sorted_Classifier_Images/Model_Training"

LABEL_SOURCE_MODE = "gcs_export"  # "label_studio_api", "gcs_export", or "user_upload_zip"

LABEL_EXPORT_GCS_URI = "gs://REPLACE_BUCKET/REPLACE_PATH/label-studio-yolo-export.zip"

USER_LABEL_ZIP = Path.cwd() / "uploads" / "label-studio-yolo-export.zip"

LABEL_EXPORT_UPLOAD_URI = ""  # Optional GCS URI for mirroring a fresh API export



LABEL_STUDIO_URL = "https://REPLACE_WITH_LABEL_STUDIO_HOST"

LABEL_STUDIO_PROJECT_ID = None  # Replace with an integer

LABEL_STUDIO_TOKEN_ENV = "LABEL_STUDIO_API_TOKEN"



WORKSPACE = Path.cwd() / "workspace" / PROJECT_NAME

RAW_IMAGES_DIR = WORKSPACE / "raw" / "images"

RAW_LABELS_DIR = WORKSPACE / "raw" / "label_export"

DATASET_DIR = WORKSPACE / "dataset"

DATASET_YAML = WORKSPACE / "dataset.yaml"

SPLIT_MANIFEST = WORKSPACE / "split_manifest.json"

PREP_SUMMARY = WORKSPACE / "prep_summary.json"



CLASS_NAMES = ["REPLACE_WITH_CLASS_0"]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

SPLIT_RATIOS = {"train": 0.70, "val": 0.20, "test": 0.10}



# Generate the seed once from the UTC system date/time. Reuse the seed recorded

# in split_manifest.json when an exact split rerun is required.

PREP_DATETIME_UTC = datetime.now(timezone.utc)

RANDOM_SEED = int(PREP_DATETIME_UTC.strftime("%Y%m%d%H%M%S")) % (2**32 - 1)

PREP_NAME = f"{PROJECT_NAME}_{PREP_DATETIME_UTC:%Y%m%dT%H%M%SZ}"



for directory in (RAW_IMAGES_DIR, RAW_LABELS_DIR, DATASET_DIR):

    directory.mkdir(parents=True, exist_ok=True)



assert LABEL_SOURCE_MODE in {"label_studio_api", "gcs_export", "user_upload_zip"}

assert math.isclose(sum(SPLIT_RATIOS.values()), 1.0)

assert 0 <= RANDOM_SEED < 2**32

print(f"Workspace: {WORKSPACE}")

print(f"Preparation name: {PREP_NAME}")

print(f"System datetime seed: {RANDOM_SEED}")

## 3. Authenticate and configure Google Cloud



Run interactive authentication in a VS Code terminal, not from the notebook, so browser and credential prompts remain visible:



```powershell

gcloud auth login

gcloud auth application-default login

gcloud config set project YOUR_PROJECT_ID

gcloud auth list

```



For a non-interactive service account, prefer workload identity on Google Cloud. If a key is unavoidable, store it outside the repository and run `gcloud auth activate-service-account --key-file PATH_TO_KEY.json`; set `GOOGLE_APPLICATION_CREDENTIALS` only in the process environment. The following cell verifies the CLI, active identity, project, and source path without displaying credentials.

In [ ]:
def run_command(arguments: list[str], *, capture: bool = False) -> subprocess.CompletedProcess[str]:

    print("Running:", subprocess.list2cmdline(arguments))

    return subprocess.run(

        arguments,

        check=True,

        text=True,

        capture_output=capture,

    )





if shutil.which("gcloud") is None:

    raise RuntimeError("gcloud was not found. Install Google Cloud CLI, restart VS Code, and rerun.")



active_account = run_command(

    ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=value(account)"],

    capture=True,

).stdout.strip()

if not active_account:

    raise RuntimeError("No active gcloud account. Run `gcloud auth login` in a terminal.")



if "REPLACE_WITH" in GCP_PROJECT_ID:

    raise ValueError("Set GCP_PROJECT_ID in the configuration cell before continuing.")



run_command(["gcloud", "config", "set", "project", GCP_PROJECT_ID])

print(f"Active account: {active_account}")

print(f"Configured project: {GCP_PROJECT_ID}")

## 4. Validate source access and sync images



The listing is a cheap permission check before a large transfer. `gcloud storage rsync --recursive` is idempotent: rerunning it copies new or changed files. It does not delete local files unless `--delete-unmatched-destination-objects` is deliberately added.



For very large paths, test a smaller prefix first or add supported `gcloud storage rsync` include/exclude flags after checking your installed CLI version.

In [ ]:
try:

    listing = run_command(

        ["gcloud", "storage", "ls", "--limit=20", IMAGE_BUCKET_URI],

        capture=True,

    ).stdout

except subprocess.CalledProcessError as error:

    message = error.stderr.strip() if error.stderr else str(error)

    raise RuntimeError(f"Cannot list source data. Check the URI, active account, and IAM access.\n{message}") from error



print("Sample source objects:")

print(listing or "No objects found at this exact prefix.")



run_command([

    "gcloud", "storage", "rsync", "--recursive",

    IMAGE_BUCKET_URI, str(RAW_IMAGES_DIR),

])



downloaded_images = sorted(

    path for path in RAW_IMAGES_DIR.rglob("*")

    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

)

if not downloaded_images:

    raise RuntimeError(f"No supported images were synced to {RAW_IMAGES_DIR}")

print(f"Synced image files: {len(downloaded_images):,}")

## 5. Retrieve YOLO labels



Choose one source in the configuration cell:



1. **Label Studio API:** set `LABEL_STUDIO_URL`, `LABEL_STUDIO_PROJECT_ID`, and the `LABEL_STUDIO_API_TOKEN` environment variable. The token is sent only in the HTTPS `Authorization: Token ...` header.

2. **GCS export:** set `LABEL_EXPORT_GCS_URI` to a Label Studio YOLO ZIP object or GCS directory.

3. **JupyterLab upload:** create the local `uploads` folder in the JupyterLab file browser, upload the YOLO export ZIP, set `USER_LABEL_ZIP` to its path, and select `user_upload_zip`.



A fresh API export can optionally be mirrored to `LABEL_EXPORT_UPLOAD_URI`. Every ZIP source is checked and extracted with path traversal protection. Existing raw label extraction files are cleared on each run so sources cannot be mixed accidentally.



> Label Studio export formats can differ by server version. This template requests `exportType=YOLO`; confirm that the project contains completed object-detection annotations and that `CLASS_NAMES` matches the exported class order.

In [ ]:
def safe_extract_zip(archive_path: Path, destination: Path) -> None:

    destination = destination.resolve()

    with zipfile.ZipFile(archive_path) as archive:

        for member in archive.infolist():

            target = (destination / member.filename).resolve()

            if destination not in target.parents and target != destination:

                raise ValueError(f"Unsafe ZIP member: {member.filename}")

        archive.extractall(destination)





if any(RAW_LABELS_DIR.iterdir()):

    print(f"Clearing prior raw label export: {RAW_LABELS_DIR}")

    shutil.rmtree(RAW_LABELS_DIR)

RAW_LABELS_DIR.mkdir(parents=True)



export_zip = WORKSPACE / "raw" / "label_studio_yolo_export.zip"

if export_zip.exists():

    export_zip.unlink()



if LABEL_SOURCE_MODE == "label_studio_api":

    token = os.environ.get(LABEL_STUDIO_TOKEN_ENV)

    if not token:

        raise RuntimeError(f"Set the {LABEL_STUDIO_TOKEN_ENV} environment variable and restart the kernel.")

    if LABEL_STUDIO_PROJECT_ID is None or "REPLACE_WITH" in LABEL_STUDIO_URL:

        raise ValueError("Set LABEL_STUDIO_URL and LABEL_STUDIO_PROJECT_ID.")



    headers = {"Authorization": f"Token {token}"}

    project_url = f"{LABEL_STUDIO_URL.rstrip('/')}/api/projects/{LABEL_STUDIO_PROJECT_ID}"

    project_response = requests.get(project_url, headers=headers, timeout=60)

    project_response.raise_for_status()

    print(f"Connected to Label Studio project: {project_response.json().get('title', LABEL_STUDIO_PROJECT_ID)}")



    export_response = requests.get(

        f"{project_url}/export",

        headers=headers,

        params={"exportType": "YOLO"},

        timeout=600,

    )

    export_response.raise_for_status()

    export_zip.write_bytes(export_response.content)

    print(f"Downloaded export: {export_zip} ({export_zip.stat().st_size:,} bytes)")



    if LABEL_EXPORT_UPLOAD_URI:

        run_command(["gcloud", "storage", "cp", str(export_zip), LABEL_EXPORT_UPLOAD_URI])

elif LABEL_SOURCE_MODE == "gcs_export":

    if "REPLACE_" in LABEL_EXPORT_GCS_URI:

        raise ValueError("Set LABEL_EXPORT_GCS_URI to a Label Studio YOLO export in GCS.")

    if LABEL_EXPORT_GCS_URI.lower().endswith(".zip"):

        run_command(["gcloud", "storage", "cp", LABEL_EXPORT_GCS_URI, str(export_zip)])

    else:

        run_command([

            "gcloud", "storage", "rsync", "--recursive",

            LABEL_EXPORT_GCS_URI, str(RAW_LABELS_DIR),

        ])

else:

    uploaded_zip = USER_LABEL_ZIP.expanduser().resolve()

    if not uploaded_zip.is_file():

        raise FileNotFoundError(

            f"Uploaded label ZIP not found: {uploaded_zip}. Upload it in JupyterLab and update USER_LABEL_ZIP."

        )

    if not zipfile.is_zipfile(uploaded_zip):

        raise ValueError(f"The uploaded label file is not a valid ZIP archive: {uploaded_zip}")

    shutil.copy2(uploaded_zip, export_zip)

    print(f"Using uploaded label export: {uploaded_zip}")



if export_zip.exists():

    if not zipfile.is_zipfile(export_zip):

        raise ValueError(f"Downloaded label export is not a valid ZIP archive: {export_zip}")

    safe_extract_zip(export_zip, RAW_LABELS_DIR)



raw_label_files = sorted(RAW_LABELS_DIR.rglob("*.txt"))

if not raw_label_files:

    raise RuntimeError(f"No YOLO .txt files found under {RAW_LABELS_DIR}")

print(f"Raw YOLO text files: {len(raw_label_files):,}")

## 6. Validate image/label integrity



Pairing uses each file's stem (for example, `image_001.jpg` pairs with `image_001.txt`). Duplicate stems are rejected because flattening them into a YOLO split would overwrite data. Empty label files are reported as valid negative/background images; change that policy here if your project requires every image to contain an object.



The checks fail before splitting or training when labels are missing, orphaned, malformed, non-finite, outside YOLO's normalized `[0, 1]` range, or refer to unknown class IDs. If the export contains `classes.txt`, compare its order with `CLASS_NAMES`; the configured order is authoritative.

In [ ]:
def unique_by_stem(paths: list[Path], kind: str) -> dict[str, Path]:

    grouped: dict[str, list[Path]] = {}

    for path in paths:

        grouped.setdefault(path.stem, []).append(path)

    duplicates = {stem: items for stem, items in grouped.items() if len(items) > 1}

    if duplicates:

        preview = {stem: [str(item) for item in items] for stem, items in list(duplicates.items())[:10]}

        raise ValueError(f"Duplicate {kind} stems must be resolved: {preview}")

    return {stem: items[0] for stem, items in grouped.items()}





images = sorted(

    path for path in RAW_IMAGES_DIR.rglob("*")

    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

)

candidate_labels = sorted(

    path for path in RAW_LABELS_DIR.rglob("*.txt")

    if path.name.lower() not in {"classes.txt", "notes.txt"}

)

images_by_stem = unique_by_stem(images, "image")

labels_by_stem = unique_by_stem(candidate_labels, "label")



missing_labels = sorted(set(images_by_stem) - set(labels_by_stem))

orphan_labels = sorted(set(labels_by_stem) - set(images_by_stem))

empty_labels: list[str] = []

invalid_rows: list[str] = []

class_counts: Counter[int] = Counter()



for stem in sorted(set(images_by_stem) & set(labels_by_stem)):

    label_path = labels_by_stem[stem]

    lines = [line.strip() for line in label_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]

    if not lines:

        empty_labels.append(stem)

        continue



    for line_number, line in enumerate(lines, start=1):

        fields = line.split()

        location = f"{label_path}:{line_number}"

        if len(fields) != 5:

            invalid_rows.append(f"{location}: expected 5 values, got {len(fields)}")

            continue

        try:

            raw_class_id = float(fields[0])

            values = [float(value) for value in fields[1:]]

        except ValueError:

            invalid_rows.append(f"{location}: values must be numeric")

            continue

        class_id = int(raw_class_id)

        if raw_class_id != class_id or not 0 <= class_id < len(CLASS_NAMES):

            invalid_rows.append(f"{location}: invalid class ID {fields[0]}")

        elif not all(math.isfinite(value) and 0.0 <= value <= 1.0 for value in values):

            invalid_rows.append(f"{location}: coordinates must be finite and normalized to [0, 1]")

        else:

            class_counts[class_id] += 1



classes_files = list(RAW_LABELS_DIR.rglob("classes.txt"))

if classes_files:

    exported_classes = [line.strip() for line in classes_files[0].read_text(encoding="utf-8-sig").splitlines() if line.strip()]

    print(f"Exported classes: {exported_classes}")

    print(f"Configured classes: {CLASS_NAMES}")



report = {

    "images": len(images_by_stem),

    "labels": len(labels_by_stem),

    "matched_pairs": len(set(images_by_stem) & set(labels_by_stem)),

    "missing_labels": len(missing_labels),

    "orphan_labels": len(orphan_labels),

    "empty_labels": len(empty_labels),

    "invalid_rows": len(invalid_rows),

    "instances_by_class": {CLASS_NAMES[class_id]: count for class_id, count in sorted(class_counts.items())},

}

print(json.dumps(report, indent=2))



errors = []

if missing_labels:

    errors.append(f"Missing labels ({len(missing_labels)}): {missing_labels[:20]}")

if orphan_labels:

    errors.append(f"Orphan labels ({len(orphan_labels)}): {orphan_labels[:20]}")

if invalid_rows:

    errors.append("Invalid rows:\n" + "\n".join(invalid_rows[:50]))

if errors:

    raise ValueError("Dataset validation failed.\n\n" + "\n\n".join(errors))



matched_stems = sorted(set(images_by_stem) & set(labels_by_stem))

if not matched_stems:

    raise ValueError("No valid image/label pairs were found.")

## 7. Create reproducible splits and `dataset.yaml`



This cell recreates the derived dataset from validated raw inputs. The random seed and exact assignments are saved in `split_manifest.json`, so a run can be audited and recreated. Raw GCS and Label Studio downloads are not modified.



For datasets with related frames, sites, transects, or video sequences, replace the image-level shuffle with a group-aware split to prevent leakage between training and evaluation sets.

In [ ]:
if any(name.startswith("REPLACE_WITH") for name in CLASS_NAMES):

    raise ValueError("Replace CLASS_NAMES with the Label Studio class names in exact class-ID order.")



if DATASET_DIR.exists():

    shutil.rmtree(DATASET_DIR)



for split in SPLIT_RATIOS:

    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)

    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)



shuffled_stems = matched_stems.copy()

random.Random(RANDOM_SEED).shuffle(shuffled_stems)

total = len(shuffled_stems)

train_end = int(total * SPLIT_RATIOS["train"])

val_end = train_end + int(total * SPLIT_RATIOS["val"])

assignments = {

    "train": shuffled_stems[:train_end],

    "val": shuffled_stems[train_end:val_end],

    "test": shuffled_stems[val_end:],

}



for split, stems in assignments.items():

    for stem in stems:

        image_path = images_by_stem[stem]

        label_path = labels_by_stem[stem]

        shutil.copy2(image_path, DATASET_DIR / "images" / split / image_path.name)

        shutil.copy2(label_path, DATASET_DIR / "labels" / split / f"{stem}.txt")



manifest = {

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "seed": RANDOM_SEED,

    "ratios": SPLIT_RATIOS,

    "source_images": IMAGE_BUCKET_URI,

    "label_source_mode": LABEL_SOURCE_MODE,

    "classes": CLASS_NAMES,

    "assignments": assignments,

}

(WORKSPACE / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")



dataset_config = {

    "path": DATASET_DIR.resolve().as_posix(),

    "train": "images/train",

    "val": "images/val",

    "test": "images/test",

    "names": {index: name for index, name in enumerate(CLASS_NAMES)},

}

DATASET_YAML.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")



print(yaml.safe_dump(dataset_config, sort_keys=False))

print("Split counts:", {split: len(stems) for split, stems in assignments.items()})

if not assignments["train"] or not assignments["val"]:

    raise ValueError("Training and validation splits must both contain data; adjust ratios or add examples.")

## 8. Inspect labeled samples



Review several images after splitting and before spending GPU time. Boxes are drawn from YOLO's normalized `class x_center y_center width height` format. Empty-label background images display without boxes.

In [ ]:
sample_paths = sorted((DATASET_DIR / "images" / "train").iterdir())[:9]

figure, axes = plt.subplots(3, 3, figsize=(15, 15))



for axis, image_path in zip(axes.flat, sample_paths):

    with Image.open(image_path) as source:

        image = source.convert("RGB")

    draw = ImageDraw.Draw(image)

    width, height = image.size

    label_path = DATASET_DIR / "labels" / "train" / f"{image_path.stem}.txt"



    for line in label_path.read_text(encoding="utf-8-sig").splitlines():

        if not line.strip():

            continue

        class_id, x_center, y_center, box_width, box_height = map(float, line.split())

        left = (x_center - box_width / 2) * width

        top = (y_center - box_height / 2) * height

        right = (x_center + box_width / 2) * width

        bottom = (y_center + box_height / 2) * height

        draw.rectangle((left, top, right, bottom), outline="red", width=max(2, width // 500))

        draw.text((left, max(0, top - 12)), CLASS_NAMES[int(class_id)], fill="red")



    axis.imshow(image)

    axis.set_title(image_path.name, fontsize=9)

    axis.axis("off")



for axis in axes.flat[len(sample_paths):]:

    axis.axis("off")



plt.tight_layout()

plt.show()

## 9. Publish the training handoff



The training notebook consumes only the prepared workspace. This final check confirms the three contract files exist and writes content hashes so training runs can identify the exact split and YAML configuration they used.



Handoff artifacts:



- `dataset.yaml`: Ultralytics dataset paths and class mapping.

- `split_manifest.json`: source references, datetime seed, split ratios, and exact assignments.

- `prep_summary.json`: preparation identity, counts, and hashes used by notebook 2.

In [ ]:
def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for block in iter(lambda: file.read(1024 * 1024), b""):

            digest.update(block)

    return digest.hexdigest()





for required_path in (DATASET_YAML, SPLIT_MANIFEST):

    if not required_path.is_file():

        raise FileNotFoundError(f"Missing preparation artifact: {required_path}")



prep_summary = {

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "prep_name": PREP_NAME,

    "workspace": str(WORKSPACE.resolve()),

    "dataset_yaml": str(DATASET_YAML.resolve()),

    "split_manifest": str(SPLIT_MANIFEST.resolve()),

    "dataset_yaml_sha256": sha256_file(DATASET_YAML),

    "split_manifest_sha256": sha256_file(SPLIT_MANIFEST),

    "seed": RANDOM_SEED,

    "classes": CLASS_NAMES,

    "split_counts": {split: len(stems) for split, stems in assignments.items()},

}

PREP_SUMMARY.write_text(json.dumps(prep_summary, indent=2), encoding="utf-8")

print(json.dumps(prep_summary, indent=2))

print("Dataset preparation is complete. Continue with 02_cloud_yolo_training.ipynb.")